<a href="https://colab.research.google.com/github/Moquiuti/NLP/blob/main/avaliandoUmProduto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
!pip install unidecode

In [16]:
import nltk

nltk.download('rslp')

[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Package rslp is already up-to-date!


True

In [18]:
from nltk import tokenize
import nltk
from unidecode import unidecode

def processar_avaliacao(avaliacao):
    # passo 1
    token_pontuacao = tokenize.WordPunctTokenizer()
    tokens = token_pontuacao.tokenize(avaliacao)
    palavras_irrelevantes = nltk.corpus.stopwords.words('portuguese')


    # passo 2
    frase_processada = [
        palavra for palavra in tokens
        if palavra.lower() not in palavras_irrelevantes
    ]

    # passo 3
    frase_processada = [
        palavra for palavra in frase_processada
        if palavra.isalpha()
    ]

    # passo 4
    frase_processada = [
        unidecode.unidecode(palavra)
        for palavra in frase_processada
    ]

    stemmer = nltk.RSLPStemmer()

    # passo 5
    frase_processada = [
        stemmer.stem(palavra)
        for palavra in frase_processada
    ]

    return ' '.join(frase_processada)

In [22]:
import pandas as pd
import nltk
import unidecode

from nltk import tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

nltk.download('stopwords')
nltk.download('rslp')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Package rslp is already up-to-date!


True

In [24]:
df = pd.read_csv('dataset_avaliacoes.csv')

palavras_irrelevantes = nltk.corpus.stopwords.words('portuguese')
token_pontuacao = tokenize.WordPunctTokenizer()
stemmer = nltk.RSLPStemmer()

In [25]:
def processar_avaliacao(avaliacao):
    # passo 1
    tokens = token_pontuacao.tokenize(str(avaliacao))

    # passo 2
    frase_processada = [
        palavra for palavra in tokens
        if palavra.lower() not in palavras_irrelevantes
    ]

    # passo 3
    frase_processada = [
        palavra for palavra in frase_processada
        if palavra.isalpha()
    ]

    # passo 4
    frase_processada = [
        unidecode.unidecode(palavra)
        for palavra in frase_processada
    ]

    # passo 5
    frase_processada = [
        stemmer.stem(palavra)
        for palavra in frase_processada
    ]

    return ' '.join(frase_processada)

In [26]:
df["tratamento_5"] = df["avaliacao"].apply(processar_avaliacao)

tfidf_1000 = TfidfVectorizer(
    lowercase=False,
    max_features=1000,
    ngram_range=(1, 2)
)

vetor_tfidf_1000 = tfidf_1000.fit_transform(df["tratamento_5"])

X_treino, X_teste, y_treino, y_teste = train_test_split(
    vetor_tfidf_1000,
    df["sentimento"],
    random_state=4978
)

regressao_logistica = LogisticRegression(max_iter=1000)
regressao_logistica.fit(X_treino, y_treino)

acuracia = regressao_logistica.score(X_teste, y_teste)

print(f"Acurácia do modelo: {acuracia * 100:.2f} %")

Acurácia do modelo: 91.82 %


In [27]:
url = "https://raw.githubusercontent.com/alura-cursos/nlp_analise_sentimento/refs/heads/main/Dados/novas_avaliacoes.csv"

novas_avaliacoes = pd.read_csv(url)

novas_avaliacoes["avaliacao_tratada"] = novas_avaliacoes["avaliacao"].apply(processar_avaliacao)

matriz_novas_avaliacoes = tfidf_1000.transform(
    novas_avaliacoes["avaliacao_tratada"]
)

sentimentos_previstos = regressao_logistica.predict(matriz_novas_avaliacoes)

resultado = pd.DataFrame({
    "Avaliação": novas_avaliacoes["avaliacao"],
    "Sentimento Previsto": sentimentos_previstos
})

resultado

,Avaliação,Sentimento Previsto
0,Até que o sabor é bom. Muito melhor que tomar ...,positivo
1,"Nossa sério, absurdo, compro o produto com rec...",negativo
2,Bom custo benefício,positivo
3,Saboroso e nutritivo. Tem me salvado nos dias ...,positivo
4,"Fiquei decepcionada com a situação, a embalage...",negativo
5,Uma unidade do pacote veio furada e vazou em t...,negativo
6,Comprei em junho mas o produto venceu em março,negativo
7,"Gente, o meu chegou todo certinho, eu tava com...",positivo
8,"Gosto muito desse produto, ótima qualidade, e ...",positivo
9,Esse produto veio na quantidade que eu não ped...,negativo


In [30]:
vetor_tfidf_1000 = tfidf_1000.fit_transform(df["tratamento_5"])

In [31]:
matriz_novas_avaliacoes = tfidf_1000.transform(novas_avaliacoes["avaliacao_tratada"])

In [32]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    vetor_tfidf_1000,
    df["sentimento"],
    random_state=4978
)

regressao_logistica = LogisticRegression(max_iter=1000)
regressao_logistica.fit(X_treino, y_treino)

LogisticRegression(max_iter=1000)

In [33]:
sentimentos_previstos = regressao_logistica.predict(matriz_novas_avaliacoes)

In [34]:
resultado = pd.DataFrame({
    "Avaliação": novas_avaliacoes["avaliacao"],
    "Sentimento Previsto": sentimentos_previstos
})

resultado

,Avaliação,Sentimento Previsto
0,Até que o sabor é bom. Muito melhor que tomar ...,positivo
1,"Nossa sério, absurdo, compro o produto com rec...",negativo
2,Bom custo benefício,positivo
3,Saboroso e nutritivo. Tem me salvado nos dias ...,positivo
4,"Fiquei decepcionada com a situação, a embalage...",negativo
5,Uma unidade do pacote veio furada e vazou em t...,negativo
6,Comprei em junho mas o produto venceu em março,negativo
7,"Gente, o meu chegou todo certinho, eu tava com...",positivo
8,"Gosto muito desse produto, ótima qualidade, e ...",positivo
9,Esse produto veio na quantidade que eu não ped...,negativo
